# Lucia v1.1 4B: the Apache-2.0 line

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/AbstractEyes/qwen-symbolic-thinking/blob/main/notebooks/lucia_v11_4b.ipynb)

The fine-tune of [Qwen/Qwen3.5-4B](https://huggingface.co/Qwen/Qwen3.5-4B) on rows written by outside writers, held near the base model's behavior while it learns the rows. This notebook runs the work both lines share, then the Apache-2.0 line:

1. the Apache-2.0 training set (`tools.build_rows`): the rows whose pictures and annotations are CC BY 4.0 or Apache-2.0, no row written with a photo whose own license is not given, the screen's, the probe's and the museum read's pictures held out, and 140 training pictures drawn for the held-out read;
2. the non-commercial training set: every row, the same pictures held out, the pictures kept in the writers' storage bucket copied into the twin first;
3. the general rows (`tools.anchors`): the base model's own answers to requests that do not ask for the row form;
4. a 20-step run with every keeping term on, then the read of the earlier adapter (Lucia v1 4B) against its base;
5. the four versions of the Apache-2.0 line, one after another: the plain recipe, KL to the base on the general rows, the target-masked KL, rehearsal on the general rows; each with its screen, probe, museum and preservation reads.

The non-commercial line runs in `lucia_v11_4b_nc.ipynb` on a second card; it waits for steps 2 and 3.

Runtime: a GPU (the RTX PRO 6000) and the `HF_TOKEN` secret with read access to the accepted rows and write access to the private twin `AbstractPhil/qwen-symbolic-thinking-image-training-private`, where every file of these runs lives (the training sets hold rows that are not public). Run C1, restart the session (Runtime > Restart session), then run C2. Each run leaves `runs/done/<name>.json` in the twin; the queue cell ends after 45 idle minutes.


In [ ]:
# C1: install, then Runtime > Restart session
import importlib.util, subprocess, sys

def pip(*args):
    subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", *args])

pip("git+https://github.com/AbstractEyes/qwen-symbolic-thinking@v0.1.57", "transformers>=5.0", "accelerate>=1.0", "peft>=0.18")

FAST_PATH = True   # the linear-attention kernels Qwen3.5 uses; causal-conv1d compiles for about 5 minutes; all or nothing
if FAST_PATH:
    if all(importlib.util.find_spec(m) for m in ("causal_conv1d", "fla", "triton")):
        print("fast path already present")
    else:
        try:
            pip("causal-conv1d", "--no-build-isolation")
            pip("flash-linear-attention", "triton")
            print("fast path on")
        except subprocess.CalledProcessError as e:
            print("fast path off (training still runs, slower):", e)
            subprocess.call([sys.executable, "-m", "pip", "uninstall", "-y", "-q", "causal-conv1d"])

try:                # Colab's torchao 0.10 breaks PEFT when it builds a LoRA (PEFT wants 0.16 or newer, or none at all)
    import torchao
    _v = tuple(int(x) for x in torchao.__version__.split(".")[:2])
    if _v < (0, 16):
        subprocess.call([sys.executable, "-m", "pip", "uninstall", "-y", "-q", "torchao"])
        print("torchao", torchao.__version__, "removed: PEFT cannot build a LoRA beside it")
    else:
        print("torchao", torchao.__version__, "stays")
except ImportError:
    print("no torchao")

print("installed; now Runtime > Restart session, then run C2")

In [ ]:
# C2: work through this line's queue (runs/queue_lucia_v11_4b.json in the private twin; new runs can be added while it runs)
import os
os.environ["QST_REPO"] = "AbstractPhil/qwen-symbolic-thinking-image-training-private"   # before the package is imported: every file of these runs stays private
LANE = "lucia_v11_4b"

from qwen_symbolic_thinking import REPO, __version__
from qwen_symbolic_thinking.queue import run_queue

print("package", __version__, "| project repo", REPO, "| lane", LANE)
ran = run_queue(LANE, wait_hours=12)
print("ran:", ran)
